# Beamforming to a 3D grid with `zea.Pipeline`

In this notebook, we demonstrate beamforming 3D data acquired with a matrix probe using a `zea.Pipeline`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tue-bmd/zea/blob/main/docs/source/notebooks/pipeline/3d_beamforming_example.ipynb)
&nbsp;
[![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/tue-bmd/zea/blob/main/docs/source/notebooks/pipeline/3d_beamforming_example.ipynb)
&nbsp;
[![Hugging Face dataset](https://img.shields.io/badge/Hugging%20Face-Dataset-yellow?logo=huggingface)](https://huggingface.co/datasets/zeahub/phantoms)

‼️ **Important:** This notebook is optimized for **GPU/TPU**. Code execution on a **CPU** may be very slow.

If you are running in Colab, please enable a hardware accelerator via:

**Runtime → Change runtime type → Hardware accelerator → GPU/TPU** 🚀.

In [1]:
%%capture
%pip install zea

In [2]:
import os

os.environ["KERAS_BACKEND"] = "jax"
os.environ["ZEA_DISABLE_CACHE"] = "1"

In [3]:
import zea
from zea.ops import (
    Pipeline,
    Cast,
    Demodulate,
    Map,
    EnvelopeDetect,
    ReshapeGrid,
    Normalize,
    LogCompress,
    TOFCorrection,
    DelayAndSum,
)
from zea.visualize import set_mpl_style

zea.init_device(verbose=False)
set_mpl_style()

zea: Using backend 'jax'


In [4]:
# Set rate to downscale grid resolution for more efficient beamforming
downscale_rate = 2

First, we download an RF data tensor acquired from a CIRS040 phantom using an 8MHz 32x32 element Matrix probe. We then load the first frame, which is of shape `(1, 56, 1280, 1024, 1)`, corresponding to 1 frame, 56 transmit events, with 1280 axial samples across 1024 channels, and 1 final dimension to indicate that the data is real-valued.

In [5]:
path = "hf://zeahub/phantoms/2025_12_16_cirs_focused_3d.hdf5"

with zea.File(path, mode="r", revision="v0.1.0") as file:
    rf_data = file.data.raw_data[0]
    parameters = file.load_parameters()

rf_data = rf_data[None, ...]
print(f"RF data shape = {rf_data.shape}")

RF data shape = (1, 56, 1280, 1024, 1)


Next we specify our desired beamforming parameters by modifying attributes of the `zea.Parameters` object. This defines our 3D beamforming grid, which is of shape `(254, 94, 103, 3)`, corresponding to `254` axial, `94` lateral, and `103` elevational voxels.

In [6]:
parameters.zlims = (0, 25e-3)  # reduce z-limits a bit for better visualization
parameters.grid_size_x = parameters.grid_size_x // downscale_rate
parameters.grid_size_y = parameters.grid_size_y // downscale_rate
parameters.grid_size_z = parameters.grid_size_z // downscale_rate
print(f"3D grid shape = {parameters.grid.shape}")

3D grid shape = (254, 94, 103, 3)


Next, we create a standard delay-and-sum beamforming pipeline. We use the `Map` operation to break the time-of-flight correction and summing into a number of chunks which are processed one at a time to avoid running out of GPU memory.

In [7]:
pipeline = Pipeline(
    [
        Cast(dtype="float32"),
        Demodulate(),
        Map(
            [TOFCorrection(), DelayAndSum()],
            argnames="flatgrid",
            chunks=1024,  # Increase the number of chunks if you run out of memory
        ),
        ReshapeGrid(),
        EnvelopeDetect(),
        Normalize(),
        LogCompress(),
    ],
    with_batch_dim=True,
)
inputs = pipeline.prepare_parameters(parameters)

zea: WARNING No transmit origins provided, using zeros


Finally, we can beamform and visualize the data.

In [8]:
out = pipeline(data=rf_data, **inputs)

zea: DEBUG [zea.Pipeline] The following input keys are not used by the pipeline: {'zlims', 'xlims', 'center_frequency', 'n_el'}. Make sure this is intended. This warning will only be shown once.


In [9]:
from zea.internal.notebooks import animate_volume_mip

animate_volume_mip(
    out["data"],
    f"./cirs_volume_rotation_{downscale_rate}.gif",
    n_frames=60,  # 60 frames for smooth rotation
    interval=200,  # 200ms per frame (5 fps)
    cmap="gray",
    axis=0,  # Rotate around vertical axis
    zoom=0.8,
)

zea: Successfully saved GIF to -> ./cirs_volume_rotation_2.gif


![Rotating volume](./cirs_volume_rotation_2.gif)